# Experimente ausführen
Jeder Aufruf erstellt einen eigenen Run mit frischem Modell und denselben gespeicherten Splits.
Feste Reihenfolge: **Resampling → CDA → CLP**. Jede Methode ist optional.
Bei CDA + CLP werden auch geeignete CDA-Kopien für CLP gepaart.

Paketinstallation, Repository-Clone und ggf. Drive-Mount einmalig vor diesem Notebook erledigen.
Die vorhandenen Analyse-Notebooks lesen anschließend die gespeicherten Ergebnisse.

In [13]:
from pathlib import Path
import os

# Lokal aus dem Projekt oder notebooks/ starten. Auf Colab/Kaggle ggf. explizit setzen.
repo = Path.cwd()
if not (repo / "src").is_dir():
    repo = repo.parent
if not (repo / "src").is_dir():
    raise RuntimeError("repo auf den Pfad zum gender-bias-hate-speech Repository setzen")
os.chdir(repo)
import sys
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from src.config import ExperimentConfig
from src.experiments import run_experiment
import pandas as pd

results_root = repo / "results"  # Auf Colab ggf. einen gemounteten Drive-Pfad verwenden


## Experimentplan einstellen
Hier Methoden als einfache Namen eintragen. `COMBINE_METHODS=True` erzeugt
zusätzlich alle Kombinationen der ausgewählten Methoden; einzelne Methoden bleiben enthalten.
`INCLUDE_BASELINE` steuert den Lauf ohne Mitigation und lädt dafür `baseline.yaml`.

`HYPERPARAMETERS` gilt für alle Varianten einschließlich Baseline.
`METHOD_HYPERPARAMETERS` gilt nur, wenn die jeweilige Methode aktiv ist; ein dort
angegebener Parameter ersetzt für diese Variante die allgemeine Kandidatenliste.
Alle Kandidatenlisten werden als kartesisches Produkt kombiniert, anschließend mit allen Seeds.
Eine leere Parameterliste `{}` übernimmt die YAML-Werte. Jeder Parameterwert steht in einer Liste.
`OVERRIDES` setzt feste Werte vor dem Hyperparameter-Grid. Unbekannte Config-Pfade werden abgewiesen.

Für Hyperparameterwahl `RUN_TEST=False` verwenden. Der Split-Seed bleibt unverändert.
Diese Zellen planen nur; erst die Zelle unter **Training starten** trainiert Modelle.


In [ ]:
from src.config.experiment_plan import build_experiment_plan, experiment_plan_rows

# --- Auswahl ---
EXPERIMENT_NAME = "3D_combined"
METHODS = ["clp"]                 # z.B. ["resampling", "cda", "clp"]
COMBINE_METHODS = False           # True: auch alle Kombinationen
INCLUDE_BASELINE = True
SEEDS = [40]
RUN_TEST = False                  # Validation für Hyperparameterwahl

# --- Config-Dateien ---
CONFIG_FILES = {
    "model": "configs/bert_lora.yaml",
    "training": "configs/training.yaml",
    "evaluation": "configs/evaluation.yaml",
    "dataset": "configs/datasets/3D_combined.yaml",
}

# --- Feste Werte für alle Läufe ---
OVERRIDES = {
    "tokenization.max_length": 512,
    "tokenization.truncation": True,
    # "training.fp16": True,
}

# --- Gemeinsames Grid: alle Werte mit allen Varianten kombinieren ---
HYPERPARAMETERS = {
    "training.learning_rate": [1.0e-5, 2.0e-5],  # z.B. [1.0e-5, 2.0e-5]
    # "training.batch_size": [16, 32],
    # "lora.r": [8, 16],
}

# --- Zusätzliche Grids nur für aktive Methoden ---
METHOD_HYPERPARAMETERS = {
    "clp": {
        "mitigation.clp.clp_lambda": [0],  # z.B. [0.01, 0.1, 1.0]
        # "training.learning_rate": [1.0e-5],
    },
}


In [17]:
experiments = build_experiment_plan(
    name=EXPERIMENT_NAME,
    config_files=CONFIG_FILES,
    methods=METHODS,
    combine_methods=COMBINE_METHODS,
    include_baseline=INCLUDE_BASELINE,
    seeds=SEEDS,
    run_test=RUN_TEST,
    overrides=OVERRIDES,
    hyperparameters=HYPERPARAMETERS,
    method_hyperparameters=METHOD_HYPERPARAMETERS,
)

plan = pd.DataFrame(experiment_plan_rows(experiments))
grid_columns = list(dict.fromkeys([
    *HYPERPARAMETERS,
    *(path for grid in METHOD_HYPERPARAMETERS.values() for path in grid),
]))
columns = ["experiment", "training.seed", "evaluation.run_test", *grid_columns]
print(f"Geplante Läufe: {len(experiments)}")
display(plan[[column for column in columns if column in plan.columns]])
# Alle aufgelösten Einstellungen ansehen: display(plan)


Geplante Läufe: 2


,experiment,training.seed,evaluation.run_test,training.learning_rate,mitigation.clp.clp_lambda
0,3D_combined_baseline_hp001_seed40,40,False,0.00001,NaN
1,3D_combined_clp_hp001_seed40,40,False,0.00001,0.0


## Training starten
Diese Zelle startet alle ausgewählten Läufe nacheinander. Erneutes Ausführen erzeugt neue Run-Verzeichnisse.
Bei einem Fehler stoppt die Schleife; bereits abgeschlossene Runs bleiben gespeichert.
Die Ursache steht auch in `status.json` des fehlgeschlagenen Runs.

In [ ]:
results = []
for name, config in experiments:
    result = run_experiment(config, name, results_root=results_root)
    results.append(result)
    print("Abgeschlossen:", result["run_dir"])


In [ ]:
summary = pd.DataFrame([
    {"experiment": result["experiment_name"], "run_dir": result["run_dir"],
     **result["validation_metrics"]}
    for result in results
])
display(summary)


## Gespeicherte Ergebnisse
Jeder Run enthält `config.yaml`, Split-Snapshots, `mitigation_report.json`, `training_rows.csv`,
Modell/Tokenizer, Trainingshistorie und Validation-Metriken. Mit `RUN_TEST=True` entstehen zusätzlich
Original-/Swap-Predictions, gepaarte Predictions und die Gruppen-Evaluation.
Bootstrap und ausführliche Counterfactual-Auswertung bleiben im Analyse-Notebook.

`training_rows.csv`: `id` identifiziert die Trainingszeile, `source_id` ihren direkten Vorgänger,
`root_id` die Zeile vor allen Mitigations. CLP ergänzt keine Zeilen.
Bei CLP enthält der Trainingsloss CE + Pairing, der Validation-Loss nur CE.
Die implementierte CLP-Variante unterstützt zwei Output-Logits. Equalized-Odds-Training ist noch nicht implementiert.